# DMOSpeech 2: Guidance-Decomposed Trajectory Telemetry & Risk-Controlled Handover (`GPC`, EXP-019)

**Tests the Exact Euler Trajectory Curvature Identity, guidance-decomposed drift, and the Guidance-Pareto-Cascade (`GPC`) policy on the Tesla T4 GPU:**

1. **Exact Euler Trajectory Curvature Identity (Lemma 1):**
   $$\hat{x}_1(t_j) - \hat{x}_1(t_{j-1}) = (1 - t_j)\bigl(v(x_{t_j}, t_j) - v(x_{t_{j-1}}, t_{j-1})\bigr)$$
   Proves that step-to-step endpoint drift is horizon-weighted trajectory curvature, not numerical estimation error.

2. **Exact Quadratic Guidance Decomposition ($0$ extra forward passes):**
   $$d_j^2 = p^c_j + p^g_j + p^x_j$$
   Separates conditional velocity updates ($p^c_j$) from classifier-free guidance acceleration ($p^g_j$) and cross-interaction ($p^x_j$).

3. **Guidance-Pareto-Cascade (`GPC`):**
   Combines the selected internal guidance signal ($A_k / gc_0$ or $\sum \sqrt{p^g_j}$) with a Pareto marginal-efficiency path and split-path monotone Conformal Risk Control (CRC, $\alpha = 0.05$).

4. **Confirmation on 40 Fresh Speakers & OOD Degradations:**
   Evaluates the frozen policy once on development speakers 61–100 (uninspected by EXP-012..EXP-018) and on unseen acoustic degradations (reverberation, telephone bandpass, babble noise).

### Step 1: Check GPU Runtime (Tesla T4 GPU Required)

In [ ]:
import torch
assert torch.cuda.is_available(), 'CUDA not available. Switch runtime to T4 GPU.'
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name)
assert 'T4' in gpu_name, f'Expected Tesla T4 GPU (found {gpu_name}).'

### Step 2: Clone Repositories (`rhnl-3` branch), Install Dependencies & Checkpoints

In [ ]:
import os, subprocess, hashlib, sys
THESIS_BRANCH = 'rhnl-3'
if not os.path.exists('/content/Final-Thesis-Setup/.git'):
    subprocess.run(['git', 'clone', '-b', THESIS_BRANCH, 'https://github.com/Shonget-Thesis/Final-Thesis-Setup.git', '/content/Final-Thesis-Setup'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'fetch', 'origin', THESIS_BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'checkout', '-B', THESIS_BRANCH, f'origin/{THESIS_BRANCH}'], check=True)

DMO_COMMIT = '8a7868fc3345702a586173447f4869205b501f26'
if not os.path.exists('/content/Shonget-DMOSpeech2'):
    !git clone https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git /content/Shonget-DMOSpeech2
%cd /content/Shonget-DMOSpeech2
!git fetch -q origin && git checkout -q {DMO_COMMIT}

!pip install -q "numpy==2.2.3" "pandas==2.2.3" audioop-lts
!pip install -q -r requirements.txt
!pip install -q torchdiffeq vocos speechbrain transformers jiwer soundfile librosa pyarrow scipy jieba accelerate huggingface_hub

!mkdir -p ckpts
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt -O ckpts/model_85000.pt
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt -O ckpts/model_1500.pt

for fname, digest in {
    'model_85000.pt': 'e143188f7c46b69abb60786937d79b89dac9777d3c4c1e91a97634bcd39ebf2c',
    'model_1500.pt': '6bd50428f3b5f0f25c1a60a0053f40ee47c817bda03c8ee120100f22680ae4c2'
}.items():
    h = hashlib.sha256()
    with open(f'ckpts/{fname}', 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            h.update(block)
    assert h.hexdigest() == digest, f'{fname}: SHA-256 mismatch'
    print(f'{fname}: SHA-256 verified')

test_file = 'data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav'
if not os.path.exists(test_file) or os.path.getsize(test_file) < 1000:
    !rm -rf data/seed-tts-eval
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='zhaochenyang20/seed-tts-eval', repo_type='dataset', local_dir='data/seed-tts-eval')
print('Setup and data ready.')

### Step 3: Run Smoke Test on 2 Development Speakers

In [ ]:
import os, subprocess, sys
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
os.chdir('/content/Shonget-DMOSpeech2')
OUT = '/content/drive/MyDrive/dmospeech2_results/exp019_guidance_work'
os.makedirs(OUT, exist_ok=True)

cmd = [sys.executable, '-u', '/content/Final-Thesis-Setup/code/colab/run_guidance_work_prelim.py', 'smoke', '--out', OUT, '--dmo_root', '/content/Shonget-DMOSpeech2']
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
for line in proc.stdout:
    print(line, end='', flush=True)
assert proc.wait() == 0, f'Smoke test failed with code {proc.returncode}'

### Step 4: Run Telemetry on Development Set (60 Speakers × 6 Conditions, Teacher-Only, ~1.2 Hours)

In [ ]:
cmd = [sys.executable, '-u', '/content/Final-Thesis-Setup/code/colab/run_guidance_work_prelim.py', 'dev', '--out', OUT, '--dmo_root', '/content/Shonget-DMOSpeech2']
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
for line in proc.stdout:
    print(line, end='', flush=True)
assert proc.wait() == 0, f'Dev telemetry failed with code {proc.returncode}'

### Step 5: Run Confirmation on 40 Fresh Speakers (Steps 1..8 + Telemetry, Resumable)

In [ ]:
cmd = [sys.executable, '-u', '/content/Final-Thesis-Setup/code/colab/run_guidance_work_prelim.py', 'fresh', '--out', OUT, '--dmo_root', '/content/Shonget-DMOSpeech2']
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
for line in proc.stdout:
    print(line, end='', flush=True)
assert proc.wait() == 0, f'Fresh confirmation failed with code {proc.returncode}'

### Step 6: Run Zero-Shot OOD Generalization (40 Fresh Speakers × Reverb0.3/0.9 & Telephone)

In [ ]:
cmd = [sys.executable, '-u', '/content/Final-Thesis-Setup/code/colab/run_guidance_work_prelim.py', 'ood', '--out', OUT, '--dmo_root', '/content/Shonget-DMOSpeech2']
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
for line in proc.stdout:
    print(line, end='', flush=True)
assert proc.wait() == 0, f'OOD run failed with code {proc.returncode}'

### Step 7: Analyze & Display Comprehensive Report (`GUIDANCE_WORK_REPORT.md`)

In [ ]:
from IPython.display import Markdown, display
subprocess.run([sys.executable, '/content/Final-Thesis-Setup/code/colab/run_guidance_work_prelim.py', 'analyze', '--out', OUT], check=True)
with open(f'{OUT}/GUIDANCE_WORK_REPORT.md', encoding='utf-8') as f:
    display(Markdown(f.read()))

### Step 8: Package & Download All Results Archive (`EXP019_Guidance_Work_Results.zip`)

In [ ]:
import shutil
from google.colab import files
zip_local = shutil.make_archive('/content/EXP019_Guidance_Work_Results', 'zip', OUT)
zip_drive = '/content/drive/MyDrive/dmospeech2_results/EXP019_Guidance_Work_Results.zip'
shutil.copy2(zip_local, zip_drive)
print(f'Saved archive to Google Drive: {zip_drive} ({os.path.getsize(zip_local) / (1024*1024):.2f} MB)')
files.download(zip_local)